In [1]:
%pip install numpy pandas torch sklearn tqdm

Defaulting to user installation because normal site-packages is not writeable
  Using cached torch-2.7.1-cp313-cp313-win_amd64.whl.metadata (28 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [ ]:
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader
from tqdm import tqdm

# === Load player metadata ===
main_csv = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_ratings_original_updated.csv"
stats_folder = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_fbref_tables"
df = pd.read_csv(main_csv)
df.drop(
    columns=["Unnamed: 0", "Player URL", "Team Link", "fbref_url", "fbref_alltimestat"],
    inplace=True,
)


# === Encode categorical features ===
cat_cols = [
    "Name",
    "fbref_name",
    "Team",
    "Positions",
    "Nationality",
    "Instagram",
    "Position_fbref",
    "Birth_place",
    "Recognitions",
]
encoders = {col: LabelEncoder().fit(df[col].astype(str)) for col in cat_cols}
for col in cat_cols:
    df[col] = encoders[col].transform(df[col].astype(str))

# === Normalize numeric features ===
num_cols = ["Age", "Weight", "Height"]
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# === Target columns ===
target_cols = ["Rating", "Potential", "Value", "Weekly_wage"]
y = df[target_cols].values


# === Aggregate player stats from individual folders ===
def load_player_stats(folder_path, player_id, name):
    folder_name = f"{name}_{player_id}"
    full_path = os.path.join(folder_path, folder_name)
    if not os.path.isdir(full_path):
        return pd.Series(dtype=float)

    summaries = []
    for file in os.listdir(full_path):
        if file.endswith(".csv"):
            try:
                data = pd.read_csv(os.path.join(full_path, file))
                numeric = data.select_dtypes(include=np.number)
                if not numeric.empty:
                    summaries.append(numeric.mean())
            except:
                continue

    if summaries:
        return pd.concat(summaries, axis=1).mean(axis=1)
    else:
        return pd.Series(dtype=float)


print("⏳ Aggregating per-player stats...")
stats_rows = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    stats = load_player_stats(stats_folder, row["player_id"], row["Name"])
    stats_rows.append(stats)

stats_df = pd.DataFrame(stats_rows).fillna(0)
df_combined = pd.concat(
    [df.reset_index(drop=True), stats_df.reset_index(drop=True)], axis=1
)
X = df_combined.drop(columns=target_cols).values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


class PlayerDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = PlayerDataset(X_train, y_train)
val_dataset = PlayerDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=dim, num_heads=heads, batch_first=True
        )
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class ResNetAttentionModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.res_block = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(128, input_dim)
        )
        self.transformer = TransformerBlock(input_dim)
        self.attn = nn.Sequential(
            nn.Linear(input_dim, 64), nn.Tanh(), nn.Linear(64, 1), nn.Softmax(dim=1)
        )
        self.head = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, output_dim)
        )

    def forward(self, x):
        x = self.res_block(x) + x
        x = x.unsqueeze(1)
        x = self.transformer(x)
        weights = self.attn(x)
        x = torch.sum(weights * x, dim=1)
        return self.head(x)

In [ ]:
model = ResNetAttentionModel(input_dim=X.shape[1], output_dim=y.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(1, 51):
    model.train()
    train_loss = 0
    train_loop = tqdm(train_loader, desc=f"[Epoch {epoch}] Train")
    for batch_x, batch_y in train_loop:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_loop.set_postfix(batch_loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f" Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}"
    )